In [ ]:
using DelimitedFiles;   #Paquetería para leer y exportar datos
using CairoMakie;       #Paquetería para realizar gráficas
using LaTeXStrings;     #Paquetería para escribir texto en LaTeX en las gráficas
using ConcaveHull       #Paquetería para realizar un Concave Hull de los histogramas del g(R)
using StatsBase         #Paquetería para emplear algunas funciones básicas de estadística, como el promedio

DataPath = "Quasiperiodic-Tiles/Global Structural Studies/Data/SI_Fig2_SigmaSquare_gR/"; #Ruta donde están los datos almacenados

AproxLambda(NSides) = (2π / (1 - cos(2π / NSides))); #Función que calcula nuestro aproximante estadístico

#Función que "suaviza" una gráfica reduciendo el número de puntos por su promedio (media móvil)
function promedio_m(A, i, m)
    Valores = [Factor*A[j][2] for j in (i-m):(i+m)]
    return mean(Valores)
end

promedio_m (generic function with 1 method)

In [2]:
#Diccionario con los valores del Concave Hull usado para los datos sin el inicio de las g(R)
PCH_Dict = Dict(
                5  => 331,
                7  => 442,
                9  => 466,
                11 => 800,
                13 => 718,
                15 => 554,
                17 => 398,
                19 => 282
               );

#Diccionario con los índices para eliminar el primer gran pico de la g(R)
Start_Index_Dict = Dict(
                        5  => 1,
                        7  => 1,
                        9  => 1002,
                        11 => 1002,
                        13 => 1002,
                        15 => 1002,
                        17 => 1002,
                        19 => 1002
                       );

#Diccionario con los valores de la densidad numérica de los sistemas cuasiperiódicos en 2D
Rho_Dict = Dict(
                5  => 1.2328979808609704,
                7  => 1.2517957581185175,
                9  => 1.260284085456272,
                11 => 1.2645739922427748,
                13 => 1.2670366156186388,
                15 => 1.2685820147323164,
                17 => 1.2696141740269873,
                19 => 1.2703373895977548
               );

### Visualización de la $\sigma^2(R)$ y $g(R)$

In [ ]:
##########################################################################################################################################################################
#                                                                   Datos del sistema cuasiperiódico
##########################################################################################################################################################################
NSides = 5;             #Simetría rotacional de los sistemas cuasperiódicos a analizar
Radio = 500;            #Radio de las vecindades circulares
NumPasos = Int(1e5);    #Número de divisiones entre R = 0 y R = Radio
ΔStep = Radio/NumPasos; #Tamaño del salto entre radio y radio
for NSides in 5:2:19
    ##########################################################################################################################################################################
    #                                                                  Lectura de los datos de la Sigma^2
    ##########################################################################################################################################################################
    N_p = vec(readdlm(DataPath * "Torquato_NR_N$(NSides)_Alfa0P0_R$(Radio)_Step1e5_Np.csv"));
    σ2 = vec(readdlm(DataPath * "Torquato_NR_N$(NSides)_Alfa0P0_R$(Radio)_Step1e5_SigmaCuadrada.csv"));
    ##########################################################################################################################################################################
    #                                                  Cálculo de la densidad promedio y factor de normalización (Torquato)
    ##########################################################################################################################################################################
    Rho = Rho_Dict[NSides]; #Densidad numérica de sitios a través del diccionario
    FN = 2*sqrt(π*Rho);     #Factor de normalización para mantener los resultados independientes de la densidad de puntos
    println("La densidad numérica del sistema es $(Rho)")
    ##########################################################################################################################################################################
    #                                                           Cálculo del arreglo de radios para la sigma^2
    ##########################################################################################################################################################################
    #Generamos el intervalo con los valores de la R asociados a los datos de sigma cuadrada (Incluye factor de 2*sqrt(π*Rho)
    #necesario para mantener densidad de puntos constantes en decorado, independientemente de la simetría rotacional)
    R = ΔStep:ΔStep:Radio;
    R = FN .* R;
    println("El tamaño del salto ΔR para σ^2 con N = $(NSides) es $(ΔStep)")
    ##########################################################################################################################################################################
    #                                                               Cálculo de la longitud de escala λ
    ##########################################################################################################################################################################
    λ = AproxLambda(NSides); #Longitud de escala de nuestro sistema
    println("La longitud de escala λ del sistema es $(λ)")
    ##########################################################################################################################################################################
    #                                                                    Gráfica de la σ^2(R)    
    ##########################################################################################################################################################################
    Radio_Viz = Int(ceil(6.1 * λ));
    # --- Definición de las características del lienzo y las subgráficas en él ---
    Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
    Sigma2_Ax = Axis(
                     Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                     title = L"N = %$(NSides)",                                   #Título de la gráfica
                     ylabel = L"\sigma^{2}(R)/R",                                 #Etiqueta que aparece en el eje vertical
                     titlesize = 55,                                              #Tamaño del título
                     xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
                     ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
                     xticklabelsize = 40,                                         #Tamaño para el eje X
                     yticklabelsize = 40,                                         #Tamaño para el eje Y
                     xticksize = 25,                                              #Tamaño de los ticks horizontales
                     yticksize = 25,                                              #Tamaño de los ticks verticales
                     limits = ((0, Radio_Viz), nothing),                          #Límites de la visualización para la gráfica
                    )
    hidespines!(Sigma2_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
    hidedecorations!(
                    Sigma2_Ax,
                    label = false,           #Se oculta o no las etiquetas a los ejes
                    ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                    ticks = false            #Se oculta o no los ticks de los ejes
                    )

    # --- Gráfica de los datos de la σ^2(R) ---
    Final = 0; #Datos del final a eliminar (por errores en inconsistencias de tamaño en vecindades)
    lines!(Sigma2_Ax, R[1:(end - Final)], (σ2./R)[1:(end - Final)])
    # --- Líneas verticales con los múltiplos de la longitud de escala
    vlines!(
            Sigma2_Ax, [i*λ for i in 1:Int(floor((FN*Radio)/λ))],
            linestyle = :dash,
            alpha = 1,
            linewidth = 8,
            color = :red,
            label = L"\lambda = \frac{2 \pi}{1 - \cos \left( \frac{2 \pi}{%$(NSides)} \right)} \approx %$(round(λ, digits = 2))",
           )
    # --- Líneas verticales con los múltiplos de la longitud de escala
    κ_N = (NSides/4)*FN;
    vlines!(
            Sigma2_Ax, [1*κ_N],
            linestyle = :dot,
            alpha = 1,
            linewidth = 8,
            color = :black,
            label = L"\kappa \approx %$(NSides)",
           )
    ##########################################################################################################################################################################
    #                                                                  Lectura de los datos de la g(R)
    ##########################################################################################################################################################################
    # --- Datos de las simulaciones realizadas para recolectar los datos ---
    Vecindades = 500;   #Número de vecindades por notebook
    Notebooks = 20;     #Número de notebooks

    PesosHist = vec(readdlm(DataPath * "Histograma_Pesos_N$(NSides)_Muestra$(Vecindades)_R$(Radio)_S0P001_1.csv")); #Cargamos el primer archivo con los datos
    for Nb in 2:Notebooks
        PesosHist .+= vec(readdlm(DataPath * "Histograma_Pesos_N$(NSides)_Muestra$(Vecindades)_R$(Radio)_S0P001_$(Nb).csv")); #Sumamos los datos de los siguientes archivos
    end
    PesosHist = PesosHist ./ (Notebooks * Vecindades); #Calculamos el promedio de los datos que formaron al arreglo PesosHist
    ##########################################################################################################################################################################
    #                                                             Cálculo del arreglo de radios para la g(R)
    ##########################################################################################################################################################################
    ΔR = 0.001;             #Tamaño del crecimiento en el radio de la ventana circular
    Rango = 0.0:ΔR:Radio;   #El intervalo de radios R empleados para generar los datos
    ##########################################################################################################################################################################
    #                                                    Cálculo del factor de normalización (tendencia a 1 y Torquato)
    ##########################################################################################################################################################################
    #NOTA: La variable "Factor" se compone de dos términos. El primero (1/Δ) es básicamente el inverso del 'Step' empleado en la variación del R para generar los datos originales 
    #de los que se obtuvo la altura del histograma contenido en los archivos.
    #El segundo término (1/100) se emplea para "contrarrestar" un factor de '100' que se introduce más adelante a la altura de los histogramas 'PesosHist', este factor de '100' 
    #se usa para optimizar el cálculo de las envolventes con la función ConcaveHull y no se requiere para ninguna otra función.
    Factor = (1/ΔR)/100; #Factor de normalización para las alturas del histograma a fin de que cuando R -> Inf, este tienda a 1

    #NOTA: El Factor de Normalización que introduce Torquato en su artículo sobre Hiperuniformidad (aparece en el Suplemento) toma
    #la forma 'Factor_Normalizacion = 1/(2*sqrt(π*Rho))'. El inverso de este Factor es el que se requiere al multiplicar las distancias, motivo por el que aparece dicha expresión en la normalización del Rango.
    Rango = [(Rango[i+1] + Rango[i])/2 for i in 1:(length(Rango) - 1)]; #Arreglo con los valores centrales de cada bin del histograma
    RangoNorm = 2*sqrt(π*Rho)*Rango;                                    #Re-escalamos las longitudes con el factor de Torquato
    Frecuencia = 100*(PesosHist .+ 1e-6)./(2*π*Rho*Rango);              #Frecuencias de los histogramas normalizados
    println("El tamaño del salto ΔR para g(R) con N = $(NSides) es $(ΔR)")
    ##########################################################################################################################################################################
    #                                                                    Gráfica de la g(R)    
    ##########################################################################################################################################################################
    # --- Detalles de la gráfica ---
    gR_OG_Ax = Axis(
                    Fig[2, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                    xlabel = L"R",                                               #Etiqueta que aparece en el eje horizontal
                    ylabel = L"g(R)",                                            #Etiqueta que aparece en el eje vertical
                    titlesize = 55,                                              #Tamaño del título
                    xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
                    ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
                    xticklabelsize = 40,                                         #Tamaño para el eje X
                    yticklabelsize = 40,                                         #Tamaño para el eje Y
                    xticksize = 25,                                              #Tamaño de los ticks horizontales
                    yticksize = 25,                                              #Tamaño de los ticks verticales
                    limits = ((0, Radio_Viz), nothing),                          #Límites de la visualización para la gráfica
                   )
    hidespines!(gR_OG_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
    hidedecorations!(
                     gR_OG_Ax,
                     label = false,           #Se oculta o no las etiquetas a los ejes
                     ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                     ticks = false            #Se oculta o no los ticks de los ejes
                    )
    # --- Gráfica de los datos de la g(R) ---
    EndIndex(R) = Int(ceil(R/(2*sqrt(π*Rho)*ΔR)))
    DesiredRadius = Radio_Viz;      #Radio máximo al que se quiere realizar la visualización (Tras la normalización de Torquato)
    SI = Start_Index_Dict[NSides];  #Índice del arreglo de radios que indica el sitio del arreglo en el que se comienza a graficar y analizar los datos
    EI = EndIndex(DesiredRadius);   #Índice del arreglo de radios que indica el sitio del arreglo en el que se alcanza el Radio deseado (Tras la normalización de Torquato)

    lines!(gR_OG_Ax, RangoNorm[SI:EI], Factor .* Frecuencia[SI:EI])
    # --- Líneas verticales con los múltiplos de la longitud de escala λ ---
    vlines!(
            gR_OG_Ax, [i*(2*λ) for i in 1:Int(floor((FN*Radio)/(2*λ)))],
            linestyle = :dash,
            alpha = 1,
            linewidth = 8,
            color = :red,
            label = L"\lambda = 2 \frac{2 \pi}{1 - \cos \left( \frac{2 \pi}{%$(NSides)} \right)} \approx %$(round(2*λ, digits = 2))",
           )
    # --- Líneas verticales con los múltiplos de la longitud de escala κ ---
    vlines!(
            gR_OG_Ax, [i*(2*κ_N) for i in 1:Int(floor((2*λ)/(2*κ_N)))],
            linestyle = :dot,
            alpha = 1,
            linewidth = 8,
            color = :black,
            label = L"2 \kappa \approx %$(2 * NSides)",
           )
    ###################################################################################################################
    #                                            Guardamos las gráficas    
    ###################################################################################################################
    Fig
end

### Visualización del inicio de las g(R)

In [ ]:
##########################################################################################################################################################################
#                                                                   Datos del sistema cuasiperiódico
##########################################################################################################################################################################
NSides = 5;             #Simetría rotacional de los sistemas cuasperiódicos a analizar
Radio = 500;            #Radio de las vecindades circulares
NumPasos = Int(1e5);    #Número de divisiones entre R = 0 y R = Radio
ΔStep = Radio/NumPasos; #Tamaño del salto entre radio y radio
for NSides in 5:2:19
    ##########################################################################################################################################################################
    #                                                  Cálculo de la densidad promedio y factor de normalización (Torquato)
    ##########################################################################################################################################################################
    Rho = Rho_Dict[NSides]; #Densidad numérica de sitios a través del diccionario
    println("La densidad numérica del sistema es $(Rho)")
    ##########################################################################################################################################################################
    #                                                                    Detalles del lienzo a graficar    
    ##########################################################################################################################################################################
    # --- Definición de las características del lienzo y las subgráficas en él ---
    Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
    ##########################################################################################################################################################################
    #                                                                    Lectura de los datos de la g(R)
    ##########################################################################################################################################################################
    # --- Datos de las simulaciones realizadas para recolectar los datos ---
    Vecindades = 500;   #Número de vecindades por notebook
    Notebooks = 20;     #Número de notebooks

    PesosHist = vec(readdlm(DataPath * "Histograma_Pesos_N$(NSides)_Muestra$(Vecindades)_R$(Radio)_S0P001_1.csv")); #Cargamos el primer archivo con los datos
    for Nb in 2:Notebooks
        PesosHist .+= vec(readdlm(DataPath * "Histograma_Pesos_N$(NSides)_Muestra$(Vecindades)_R$(Radio)_S0P001_$(Nb).csv")); #Sumamos los datos de los siguientes archivos
    end
    PesosHist = PesosHist ./ (Notebooks * Vecindades); #Calculamos el promedio de los datos que formaron al arreglo PesosHist
    ##########################################################################################################################################################################
    #                                                             Cálculo del arreglo de radios para la g(R)
    ##########################################################################################################################################################################
    ΔR = 0.001;             #Tamaño del crecimiento en el radio de la ventana circular
    Rango = 0.0:ΔR:Radio;   #El intervalo de radios R empleados para generar los datos
    ##########################################################################################################################################################################
    #                                                    Cálculo del factor de normalización (tendencia a 1 y Torquato)
    ##########################################################################################################################################################################
    #NOTA: La variable "Factor" se compone de dos términos. El primero (1/Δ) es básicamente el inverso del 'Step' empleado en la variación del R para generar los datos originales 
    #de los que se obtuvo la altura del histograma contenido en los archivos.
    #El segundo término (1/100) se emplea para "contrarrestar" un factor de '100' que se introduce más adelante a la altura de los histogramas 'PesosHist', este factor de '100' 
    #se usa para optimizar el cálculo de las envolventes con la función ConcaveHull y no se requiere para ninguna otra función.
    Factor = (1/ΔR)/100; #Factor de normalización para las alturas del histograma a fin de que cuando R -> Inf, este tienda a 1

    #NOTA: El Factor de Normalización que introduce Torquato en su artículo sobre Hiperuniformidad (aparece en el Suplemento) toma
    #la forma 'Factor_Normalizacion = 1/(2*sqrt(π*Rho))'. El inverso de este Factor es el que se requiere al multiplicar las distancias, motivo por el que aparece dicha expresión en la normalización del Rango.
    Rango = [(Rango[i+1] + Rango[i])/2 for i in 1:(length(Rango) - 1)]; #Arreglo con los valores centrales de cada bin del histograma
    RangoNorm = 2*sqrt(π*Rho)*Rango;                        #Re-escalamos las longitudes con el factor de Torquato
    Frecuencia = 100*(PesosHist .+ 1e-6)./(2*π*Rho*Rango);  #Frecuencias de los histogramas normalizados
    ##########################################################################################################################################################################
    #                                                                    Gráfica de la g(R)    
    ##########################################################################################################################################################################
    Radio_Viz = 27;
    # --- Detalles de la gráfica ---
    gR_OG_Ax = Axis(
                    Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                    title = L"N = %$(NSides)",                                   #Título de la gráfica
                    xlabel = L"R",                                               #Etiqueta que aparece en el eje horizontal
                    ylabel = L"g(R)",                                            #Etiqueta que aparece en el eje vertical
                    titlesize = 75,                                              #Tamaño del título
                    xlabelsize = 75,                                             #Tamaño de la etiqueta al eje horizontal
                    ylabelsize = 75,                                             #Tamaño de la etiqueta al eje vertical
                    xticklabelsize = 60,                                         #Tamaño para el eje X
                    yticklabelsize = 60,                                         #Tamaño para el eje Y
                    xticksize = 45,                                              #Tamaño de los ticks horizontales
                    yticksize = 45,                                              #Tamaño de los ticks verticales
                    limits = ((ΔR/2, Radio_Viz), nothing),                       #Límites de la visualización para la gráfica
                   )
    hidespines!(gR_OG_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
    hidedecorations!(
                     gR_OG_Ax,
                     label = false,           #Se oculta o no las etiquetas a los ejes
                     ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                     ticks = false            #Se oculta o no los ticks de los ejes
                    )
    # --- Gráfica de los datos de la g(R) ---
    EndIndex(R) = Int(ceil(R/(2*sqrt(π*Rho)*ΔR)))
    DesiredRadius = Radio_Viz;      #Radio máximo al que se quiere realizar la visualización (Tras la normalización de Torquato)
    SI = 1;                         #Índice del arreglo de radios que indica el sitio del arreglo en el que se comienza a graficar y analizar los datos
    EI = EndIndex(DesiredRadius);   #Índice del arreglo de radios que indica el sitio del arreglo en el que se alcanza el Radio deseado (Tras la normalización de Torquato)

    lines!(gR_OG_Ax, RangoNorm[SI:EI], Factor .* Frecuencia[SI:EI]);
    ###################################################################################################################
    #                                            Guardamos las gráficas    
    ###################################################################################################################
    Fig
end